# 365 Probabilidades - Dia #112
## Qual a probabilidade de você ter que dividir o prêmio?

**Tipo:** Cabalístico
**Data de publicação:** 2026-10-03
**Ferramenta:** Python
**Decisão analisada:** Que números marcar no volante?
**Hashtag:** #365Probabilidades #Dia112

---

### 📖 A História

Quando publiquei o dia da Mega-Sena, algumas pessoas me escreveram. E num assunto que eu
não tinha tocado, o mesmo tema apareceu mais de uma vez: **as datas.**

O aniversário do filho. A data do casamento. O dia em que alguém nasceu ou se foi.

Achei que fosse conversa. Fui olhar os dados e não era.

O sorteio continua sem memória nenhuma, como o #104 mostrou. A máquina não deve nada a
ninguém. Mas quem preenche o volante tem memória, e memória de todo mundo se parece.

---

### 📚 A conta que ninguém faz

A Mega-Sena sorteia seis números de 1 a 60. Qualquer combinação tem exatamente a mesma
chance de sair: uma em 50.063.860.

Só que o prêmio não é da combinação. É **dividido entre quem marcou aquela combinação.**

E aí entra uma coisa que não tem nada a ver com sorte. Os números de 1 a 31 cabem num
calendário. Os de 32 a 60 não cabem em nada. Se muita gente joga data, as combinações
feitas só de números baixos têm muito mais apostas em cima.

Isso é testável, e o teste não depende de ninguém confessar nada. Basta contar quantos
ganhadores cada concurso teve, e ver se os concursos sorteados com números baixos
tiveram mais gente para dividir. O modelo não vê o volante de ninguém; vê só quantos
dos seis números sorteados cabem num calendário.

---

### 🧮 O Modelo

Seis camadas, todas sobre os resultados oficiais.

1. **Taxa por grupo.** Ganhadores por real arrecadado, separando os concursos por quantos
   dos seis números sorteados eram menores ou iguais a 31, com intervalo exato de Poisson.
2. **Regressão de contagem** com deslocamento pelo logaritmo da arrecadação, porque
   concurso de arrecadação maior tem mais apostas e naturalmente mais ganhadores. Quatro
   ajustes: Poisson com correção de superdispersão (A), o mesmo com efeito de ano e
   marcação da Mega da Virada (B), Binomial Negativa (C) e Binomial Negativa com ano e
   Virada (D). O efeito de ano absorve mudanças históricas, inclusive o preço da aposta,
   que é constante dentro de quase todo ano.
3. **Reamostragem por concurso**, semente declarada, para não depender da forma da
   distribuição suposta na camada 2.
4. **Robustez:** o mesmo ajuste sem o grupo de 16 concursos que quebra a tendência.
5. **Dia e mês.** Números de 1 a 12 servem para dia e para mês. Se o calendário deixa
   duas marcas, a contagem de números ≤12 deve pesar além da contagem de números ≤31.
6. **Tradução em dinheiro:** duas apostas com exatamente a mesma chance de ganhar, e
   quanto sobra de prêmio em cada uma, calculado sob Poisson e sob Binomial Negativa,
   num concurso de arrecadação típica da amostra.

**Fonte:**
- Caixa Econômica Federal, resultados oficiais da Mega-Sena, concursos 1 a 3.005.
  Campos usados: números sorteados, data, `ganhadores_6_acertos` e `arrecadacao_total`.
  **Cópia pública, não o arquivo oficial baixado**, como no #104. A contagem de
  ganhadores existe para os 3.005 concursos; a arrecadação só a partir de 2009, e são
  esses concursos que entram no modelo.

**Nota metodológica:** o fator ×0,80 não se aplica. Contagem administrativa, não survey.

**Ponte interna, obrigatória:** o **#104** usou esta mesma base para perguntar se a
máquina tem memória. Não tem. Hoje a pergunta é se **os apostadores** têm. Têm.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 42
REAMOSTRAS = 500

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def mil(n):
    return f"{int(n):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- OS DADOS OFICIAIS ---
# Caixa Econômica Federal, concursos 1 a 3.005 (cópia pública).
# O arquivo está na mesma pasta do notebook.

import csv

ARQUIVO = 'megasena_caixa_ate_3005.csv'

def numero(x):
    x = (x or '').strip().replace('NULL', '')
    return float(x) if x else 0.0

concursos = []
with open(ARQUIVO, encoding='utf-8-sig') as f:
    for linha in csv.DictReader(f, delimiter=';'):
        bolas = sorted(int(linha[f'bola{i}']) for i in range(1, 7))
        data = linha['data_do_sorteio']
        arrec = numero(linha['arrecadacao_total'])
        concursos.append(dict(
            n=int(linha['concurso']),
            data=data,
            ano=int(data[-4:]),
            bolas=bolas,
            ganhadores=int(numero(linha['ganhadores_6_acertos'])),
            arrecadacao=arrec,
            baixos=sum(1 for b in bolas if b <= 31),   # cabem num calendário (dia)
            mes=sum(1 for b in bolas if b <= 12),      # cabem num calendário (dia E mês)
            # Mega da Virada: sorteio de 31/12. A Virada de 2025 aparece neste
            # arquivo com data 01/01/2026 (concurso 2955), com o fundo especial zerado.
            virada=data.startswith('31/12') or (data.startswith('01/01') and int(data[-4:]) >= 2010
                                               and numero(linha['acumulado_sorteio_especial_mega_da_virada']) == 0),
        ))

# Ganhadores existem para todos os concursos: estatística descritiva usa todos.
g_todos = np.array([c['ganhadores'] for c in concursos], dtype=float)

# No modelo só entram os concursos com arrecadação declarada, porque é ela que diz
# quantas apostas havia em jogo. A arrecadação começa em 2009; os 4 registros
# esparsos anteriores ficam de fora para não criar efeitos de ano com um concurso só.
base = [c for c in concursos if c['arrecadacao'] > 0 and c['ano'] >= 2009]

g = np.array([c['ganhadores'] for c in base], dtype=float)
a = np.array([c['arrecadacao'] for c in base])
k = np.array([c['baixos'] for c in base], dtype=float)
k12 = np.array([c['mes'] for c in base], dtype=float)
ano = np.array([c['ano'] for c in base])
virada = np.array([c['virada'] for c in base], dtype=float)

print("=" * 70)
print("  OS DADOS OFICIAIS")
print("=" * 70)
print(f"\n  Concursos no arquivo:                {mil(len(concursos))}")
print(f"  Acumulou (nenhum ganhador):          {mil((g_todos == 0).sum())} "
      f"({(g_todos == 0).mean() * 100:.1f}%)")
com = g_todos[g_todos > 0]
print(f"  Concursos com ao menos um ganhador:  {mil(len(com))}")
print(f"  Com UM ganhador só:                  {mil((g_todos == 1).sum())} "
      f"({(g_todos == 1).sum() / len(com) * 100:.1f}%)")
print(f"  Com dois ou mais:                    {mil((g_todos > 1).sum())} "
      f"({(g_todos > 1).sum() / len(com) * 100:.1f}%)")
print(f"  Média de ganhadores quando há algum: {vir(com.mean(), 2)}")
print(f"\n  Entram no modelo (arrecadação declarada, 2009 em diante): {mil(len(base))}")
print(f"  Ganhadores da sena nesses concursos: {mil(g.sum())}")
print(f"  Arrecadação somada:                  R$ {vir(a.sum() / 1e9)} bilhões")
print(f"  Concursos marcados como Mega da Virada: {int(virada.sum())} "
      f"({', '.join(str(c['ano']) for c in base if c['virada'])})")
print(f"\n  ×0,80: não se aplica. Contagem administrativa.")
print("=" * 70)

In [ ]:
# --- O MODELO ---
from scipy import optimize

# ── CAMADA 1 · taxa por grupo, com intervalo exato de Poisson ──
def ic_poisson(n_eventos, exposicao):
    baixo = 0.0 if n_eventos == 0 else stats.chi2.ppf(0.025, 2 * n_eventos) / 2
    alto = stats.chi2.ppf(0.975, 2 * n_eventos + 2) / 2
    return baixo / exposicao, alto / exposicao

print("=" * 70)
print("  CAMADA 1 · GANHADORES POR R$ 100 MILHÕES ARRECADADOS")
print("=" * 70)
print("\n  Separando os concursos por quantos dos seis números sorteados cabem")
print("  num calendário, ou seja, são menores ou iguais a 31.\n")
print(f"  {'≤31':>4} {'conc.':>6} {'ganh.':>6} {'taxa':>7}   {'IC 95%':>16}")
grupos = {}
for kk in range(7):
    m = k == kk
    if not m.any():
        continue
    n_g = g[m].sum()
    expo = a[m].sum() / 1e8
    lo, hi = ic_poisson(n_g, expo)
    grupos[kk] = dict(conc=int(m.sum()), ganh=int(n_g), taxa=n_g / expo, lo=lo, hi=hi)
    print(f"  {kk:>4} {int(m.sum()):>6} {int(n_g):>6} {vir(n_g / expo, 3):>7}"
          f"   [{vir(lo, 3)}; {vir(hi, 3)}]")
print(f"\n  De 1 número baixo para 6: a taxa observada vai de {vir(grupos[1]['taxa'], 2)} "
      f"para {vir(grupos[6]['taxa'], 2)}.")
print(f"  O grupo de {grupos[0]['conc']} concursos SEM nenhum número baixo destoa,")
print(f"  e o intervalo dele, [{vir(grupos[0]['lo'], 2)}; {vir(grupos[0]['hi'], 2)}],")
print(f"  cobre quase todos os outros. É amostra pequena, tratada na camada 4.")

# ── CAMADA 2 · regressão de contagem com deslocamento ──
# log(esperado de ganhadores) = X·β + log(arrecadação em R$100mi)
def poisson_irls(X, y, desloc):
    beta = np.zeros(X.shape[1])
    for _ in range(100):
        mu = np.exp(X @ beta + desloc)
        z = X @ beta + (y - mu) / mu
        XtW = X.T * mu
        novo = np.linalg.solve(XtW @ X, XtW @ z)
        if np.max(np.abs(novo - beta)) < 1e-10:
            beta = novo
            break
        beta = novo
    mu = np.exp(X @ beta + desloc)
    cov = np.linalg.inv((X.T * mu) @ X)
    phi = ((y - mu) ** 2 / mu).sum() / (len(y) - X.shape[1])
    return beta, cov, phi

def hessiana_numerica(f, x, h=1e-4):
    n = len(x)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            e_i, e_j = np.zeros(n), np.zeros(n)
            e_i[i], e_j[j] = h, h
            H[i, j] = (f(x + e_i + e_j) - f(x + e_i - e_j) - f(x - e_i + e_j) + f(x - e_i - e_j)) / (4 * h * h)
            H[j, i] = H[i, j]
    return H

def nb2_mle(X, y, desloc, beta0, hess=True):
    """Binomial Negativa (NB2): Var = μ + α μ². Parâmetros β e log α por máxima verossimilhança."""
    def nll(th):
        b, al = th[:-1], np.exp(th[-1])
        mu = np.exp(X @ b + desloc)
        r = 1 / al
        return -np.sum(stats.nbinom.logpmf(y, r, r / (r + mu)))
    th0 = np.r_[beta0, np.log(0.5)] if len(beta0) == X.shape[1] else beta0
    res = optimize.minimize(nll, th0, method='L-BFGS-B')
    if not hess:
        return res.x[:-1], None, np.exp(res.x[-1]), res
    res = optimize.minimize(nll, res.x, method='BFGS')
    H = hessiana_numerica(nll, res.x)
    cov = np.linalg.inv(H)
    return res.x[:-1], cov[:-1, :-1], np.exp(res.x[-1]), res

def resume(nome, beta_k, ep_k, extra=""):
    rt = np.exp(beta_k)
    lo, hi = np.exp(beta_k - 1.96 * ep_k), np.exp(beta_k + 1.96 * ep_k)
    print(f"  {nome:<44} {vir(rt, 3):>6}  [{vir(lo, 3)}; {vir(hi, 3)}]{extra}")
    return rt, lo, hi

desloc = np.log(a / 1e8)
anos = sorted(set(ano))
D_ano = np.column_stack([(ano == y).astype(float) for y in anos[1:]])   # base = 2009
X_A = np.column_stack([np.ones_like(k), k])
X_B = np.column_stack([np.ones_like(k), k, virada, D_ano])

print("\n" + "=" * 70)
print("  CAMADA 2 · O MODELO, CONTROLANDO PELA ARRECADAÇÃO (E PELO TEMPO)")
print("=" * 70)
print(f"\n  log(ganhadores esperados) = α + β × (números ≤31) [+ ano + Virada] + log(arrecadação)")
print(f"\n  {'ajuste':<44} {'razão':>6}  {'IC 95%'}")
print("  " + "-" * 66)

beta_A, cov_A, phi_A = poisson_irls(X_A, g, desloc)
ep_A = np.sqrt(np.diag(cov_A))[1] * np.sqrt(phi_A)
rt_A, lo_A, hi_A = resume(f"A · Poisson corrigido (φ = {vir(phi_A, 2)})", beta_A[1], ep_A)

beta_B, cov_B, phi_B = poisson_irls(X_B, g, desloc)
ep_B = np.sqrt(np.diag(cov_B))[1] * np.sqrt(phi_B)
rt_B, lo_B, hi_B = resume(f"B · A + ano + Virada (φ = {vir(phi_B, 2)})", beta_B[1], ep_B)

beta_C, cov_C, alpha_C, _ = nb2_mle(X_A, g, desloc, beta_A)
ep_C = np.sqrt(np.diag(cov_C))[1]
rt_C, lo_C, hi_C = resume(f"C · Binomial Negativa (α = {vir(alpha_C, 2)})", beta_C[1], ep_C)

beta_D, cov_D, alpha_D, _ = nb2_mle(X_B, g, desloc, beta_B)
ep_D = np.sqrt(np.diag(cov_D))[1]
rt_D, lo_D, hi_D = resume(f"D · C + ano + Virada (α = {vir(alpha_D, 2)})", beta_D[1], ep_D)
print("  " + "-" * 66)

# O modelo de referência do dia é o D: incorpora a superdispersão e o tempo.
beta, ep_q, rt, rt_lo, rt_hi, phi = beta_D, ep_D, rt_D, lo_D, hi_D, phi_A
t = beta[1] / ep_q
salto = np.exp(5 * beta[1])
salto_lo, salto_hi = np.exp(5 * (beta[1] - 1.96 * ep_q)), np.exp(5 * (beta[1] + 1.96 * ep_q))
print(f"\n  Referência do dia: modelo D. Cada número ≤31 a mais no sorteio está associado")
print(f"  a multiplicar a taxa esperada de ganhadores por {vir(rt, 3)} "
      f"[{vir(rt_lo, 3)}; {vir(rt_hi, 3)}], t = {vir(t, 1)}.")
print(f"  Os quatro ajustes ficam entre {vir(min(rt_A, rt_B, rt_C, rt_D), 2)} e "
      f"{vir(max(rt_A, rt_B, rt_C, rt_D), 2)}: o efeito não depende da distribuição")
print(f"  suposta nem do controle pelo tempo.")
print(f"\n  Segundo o modelo, passar de 1 para 6 números ≤31 (cinco a mais) corresponde a")
print(f"  {vir(salto, 2)} vezes a taxa esperada de ganhadores, IC 95% "
      f"[{vir(salto_lo, 2)}; {vir(salto_hi, 2)}]. É extrapolação do ajuste ({vir(rt, 2)}^5),"
      f" não razão observada.")
print(f"\n  Superdispersão (φ = {vir(phi_A, 2)} no Poisson simples): os ganhadores variam entre")
print(f"  concursos mais do que um Poisson explicaria. É compatível com apostas que não")
print(f"  se espalham igual pelas combinações. O modelo não diz por quê.")

# ── CAMADA 3 · reamostragem por concurso, no MODELO PRINCIPAL (D) ──
ger = np.random.default_rng(SEMENTE)
idx = np.arange(len(g))
th_quente = np.r_[beta_D, np.log(alpha_D)]      # partida quente: o ajuste completo
bs = np.empty(REAMOSTRAS)
for i in range(REAMOSTRAS):
    s_ = ger.choice(idx, size=len(idx), replace=True)
    try:
        b_, _, _, _ = nb2_mle(X_B[s_], g[s_], desloc[s_], th_quente, hess=False)
        bs[i] = b_[1]
    except (np.linalg.LinAlgError, ValueError):
        bs[i] = np.nan
bs = bs[~np.isnan(bs)]
bs_lo, bs_hi = np.exp(np.percentile(bs, [2.5, 97.5]))
print("\n" + "=" * 70)
print(f"  CAMADA 3 · REAMOSTRAGEM POR CONCURSO ({mil(len(bs))} reamostras, semente {SEMENTE})")
print("=" * 70)
print(f"\n  Modelo D reajustado em cada reamostra (Binomial Negativa, ano, Virada).")
print(f"  Razão de taxas: mediana {vir(np.exp(np.median(bs)), 3)}"
      f"  IC 95% percentil [{vir(bs_lo, 3)}; {vir(bs_hi, 3)}]")
print(f"  Não supõe a forma da distribuição dos estimadores, e chega perto do")
print(f"  intervalo analítico [{vir(rt_lo, 3)}; {vir(rt_hi, 3)}].")

# ── CAMADA 4 · robustez sem o grupo que destoa ──
m = k >= 1
beta_r, cov_r, alpha_r, _ = nb2_mle(X_B[m], g[m], desloc[m], beta_B)
ep_r = np.sqrt(np.diag(cov_r))[1]
print("\n" + "=" * 70)
print("  CAMADA 4 · SENSIBILIDADE SEM OS CONCURSOS SEM NENHUM NÚMERO BAIXO")
print("=" * 70)
print(f"\n  O grupo fica no modelo principal. Como são só {grupos[0]['conc']} concursos e a")
print(f"  estimativa deles é muito imprecisa, repeti o modelo D sem eles:")
print(f"  razão de taxas {vir(np.exp(beta_r[1]), 3)}"
      f"  IC 95% [{vir(np.exp(beta_r[1] - 1.96 * ep_r), 3)};"
      f" {vir(np.exp(beta_r[1] + 1.96 * ep_r), 3)}]")
print(f"  O efeito permanece e fica maior.")

# ── CAMADA 5 · dia e mês ──
X_E = np.column_stack([np.ones_like(k), k, k12, virada, D_ano])
beta_E, cov_E, alpha_E, _ = nb2_mle(X_E, g, desloc, np.r_[beta_B[:2], 0.0, beta_B[2:]])
ep_E = np.sqrt(np.diag(cov_E))
print("\n" + "=" * 70)
print("  CAMADA 5 · O CALENDÁRIO DEIXA DUAS MARCAS?")
print("=" * 70)
print(f"\n  Modelo D + contagem de números ≤12 (servem para dia E para mês):\n")
print(f"  {'por número ≤31 (além do efeito ≤12)':<44} "
      f"{vir(np.exp(beta_E[1]), 3):>6}  [{vir(np.exp(beta_E[1] - 1.96 * ep_E[1]), 3)}; "
      f"{vir(np.exp(beta_E[1] + 1.96 * ep_E[1]), 3)}]")
print(f"  {'por número ≤12 (acréscimo sobre ≤31)':<44} "
      f"{vir(np.exp(beta_E[2]), 3):>6}  [{vir(np.exp(beta_E[2] - 1.96 * ep_E[2]), 3)}; "
      f"{vir(np.exp(beta_E[2] + 1.96 * ep_E[2]), 3)}]")
print(f"  Um número de 1 a 12 sorteado multiplica a taxa por "
      f"{vir(np.exp(beta_E[1] + beta_E[2]), 2)}; um de 13 a 31, por {vir(np.exp(beta_E[1]), 2)}.")

# ── CAMADA 6 · tradução em dinheiro, sob Poisson e sob Binomial Negativa ──
# Previsão MARGINAL: para cada concurso comum da amostra (fora da Virada), com o
# próprio ano e a própria arrecadação, calcula-se a fatia esperada para uma
# combinação com k números ≤31; a fatia do dia é a média sobre os concursos.
ARREC_TIPICA = np.median(a)
comum = virada == 0
efeito_ano = np.r_[0.0, beta_D[3:]]                 # base 2009
efeito_ano_por_concurso = np.array([efeito_ano[anos.index(y)] for y in ano])
efeito_ano_medio = np.mean(efeito_ano_por_concurso[comum])   # só para o gráfico 2

def lambda_concursos(kk):
    """Outros ganhadores esperados, concurso a concurso, para k números ≤31."""
    return np.exp(beta_D[0] + beta_D[1] * kk + efeito_ano_por_concurso[comum]) * a[comum] / 1e8

def esperado_outros(kk):
    return lambda_concursos(kk).mean()

def fatia_poisson(lmbd):
    lmbd = np.atleast_1d(lmbd)
    return np.where(lmbd > 0, (1 - np.exp(-lmbd)) / np.where(lmbd > 0, lmbd, 1), 1.0)

def fatia_nb(lmbd, alpha):
    lmbd = np.atleast_1d(lmbd)[:, None]
    r = 1 / alpha
    xs = np.arange(0, 400)[None, :]
    pmf = stats.nbinom.pmf(xs, r, r / (r + lmbd))
    return np.sum(pmf / (1 + xs), axis=1)

def fatia_marginal(kk, modelo):
    lam = lambda_concursos(kk)
    f = fatia_poisson(lam) if modelo == 'poisson' else fatia_nb(lam, alpha_D)
    return f.mean()

l6, l1 = esperado_outros(6), esperado_outros(1)
fP6, fP1 = fatia_marginal(6, 'poisson'), fatia_marginal(1, 'poisson')
fN6, fN1 = fatia_marginal(6, 'nb'), fatia_marginal(1, 'nb')
ganho_P, ganho_N = fP1 / fP6 - 1, fN1 / fN6 - 1

print("\n" + "=" * 70)
print("  CAMADA 6 · DUAS COMBINAÇÕES COM A MESMA CHANCE DE GANHAR")
print("=" * 70)
print(f"\n  Média sobre os {int(comum.sum())} concursos comuns da amostra (fora da Virada), cada um")
print(f"  com o próprio ano e a própria arrecadação (mediana R$ {vir(ARREC_TIPICA / 1e6)} milhões).")
print(f"  A fatia depende do volume: mais apostas, mais chance de outro ganhador.\n")
print(f"  {'':34}{'só ≤31 (6)':>14}{'um ≤31 (1)':>14}{'ganho relativo':>16}")
print(f"  {'outros ganhadores esperados (média)':<34}{vir(l6, 2):>14}{vir(l1, 2):>14}")
print(f"  {'fatia esperada · Poisson':<34}{vir(fP6 * 100) + '%':>14}{vir(fP1 * 100) + '%':>14}"
      f"{'+' + vir(ganho_P * 100) + '%':>16}")
print(f"  {'fatia esperada · Binomial Negativa':<34}{vir(fN6 * 100) + '%':>14}{vir(fN1 * 100) + '%':>14}"
      f"{'+' + vir(ganho_N * 100) + '%':>16}")
# Sensibilidade ao volume: a fatia num concurso pequeno e num grande
lam_lo, lam_hi = np.percentile(a[comum], [10, 90])
print(f"\n  Sob os dois modelos: a mesma chance de acertar, e cerca de {vir(ganho_N * 100, 0)}% a mais")
print(f"  de prêmio esperado, condicionado ao acerto, pela diferença de popularidade")
print(f"  estimada entre as combinações. Em pontos: {vir((fN1 - fN6) * 100, 0)} pontos de fatia.")
for rot, aa in [("concurso pequeno (P10 da arrecadação)", lam_lo), ("concurso grande (P90)", lam_hi)]:
    l6_, l1_ = [np.exp(beta_D[0] + beta_D[1] * kk + efeito_ano_medio) * aa / 1e8 for kk in (6, 1)]
    f6_, f1_ = fatia_nb(l6_, alpha_D)[0], fatia_nb(l1_, alpha_D)[0]
    print(f"  {rot:<40} R$ {vir(aa / 1e6, 0):>4} mi: {vir(f6_ * 100)}% vs {vir(f1_ * 100)}% "
          f"(+{vir((f1_ / f6_ - 1) * 100, 0)}%)")
print("\n  ATENÇÃO, e isso vai no texto: evitar datas NÃO aumenta a chance de ganhar.")
print("  Aumenta o quanto sobra se ganhar. São coisas diferentes.")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO ---

ks = sorted(grupos)
taxas = [grupos[i]['taxa'] for i in ks]
los = [grupos[i]['lo'] for i in ks]
his = [grupos[i]['hi'] for i in ks]

# ══ GRÁFICO 1 · a taxa por grupo ══
fig1, ax1 = plt.subplots(figsize=(11, 6.5))
cores = [CINZA if i == 0 else VERDE for i in ks]
ax1.errorbar(ks, taxas, yerr=[np.array(taxas) - np.array(los), np.array(his) - np.array(taxas)],
             fmt='none', ecolor=CINZA, elinewidth=1.6, capsize=6, alpha=0.8)
for i, x in enumerate(ks):
    ax1.plot(x, taxas[i], 'o', color=cores[i], markersize=13, zorder=3)
    ax1.text(x, his[i] + 0.09, mil(grupos[x]['ganh']), ha='center', fontsize=10, color=CINZA)

ax1.set_xlabel('Quantos dos seis números sorteados cabem num calendário (≤ 31)')
ax1.set_ylabel('Ganhadores da sena por R$ 100 milhões arrecadados')
ax1.set_title('Quando o sorteio dá números de calendário, aparece mais gente para dividir\n'
              f'{mil(len(base))} concursos oficiais · barras são IC 95% exato de Poisson',
              fontsize=13, pad=15)
ax1.set_xticks(ks)
ax1.set_ylim(0, 3.0)
ax1.annotate(f'apenas {grupos[0]["conc"]} concursos:\no intervalo cobre quase tudo',
             xy=(0, his[0]), xytext=(0.55, 2.55), fontsize=10, color=CINZA,
             arrowprops=dict(arrowstyle='->', color=CINZA, lw=1.4))
ax1.text(5.3, 0.35, 'o número acima de cada ponto\né quantos ganhadores houve',
         fontsize=9.5, color=CINZA, ha='center')

plt.figtext(0.5, 0.005, 'Fonte: Caixa Econômica Federal, concursos 1 a 3.005 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-112-grafico-01-a-fenda.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · o modelo ajustado ══
fig2, ax2 = plt.subplots(figsize=(11, 6.5))
xx = np.linspace(0, 6, 200)
# Curva no ano médio dos concursos comuns; banda pela matriz de covariância COMPLETA
# do modelo D: var(log λ) = gᵀ Σ g, com g = [1, k, 0 (Virada), pesos dos anos].
pesos_ano = np.array([np.mean(ano[comum] == y) for y in anos[1:]])
def banda(kk):
    gvec = np.r_[1.0, kk, 0.0, pesos_ano]
    centro = gvec @ beta_D
    ep_c = np.sqrt(gvec @ cov_D @ gvec)
    return np.exp(centro), np.exp(centro - 1.96 * ep_c), np.exp(centro + 1.96 * ep_c)
prev, lo_c, hi_c = (np.array(v) for v in zip(*[banda(x_) for x_ in xx]))
ax2.fill_between(xx, lo_c, hi_c, color=VERDE, alpha=0.18)
ax2.plot(xx, prev, color=VERDE, linewidth=2.5, label='Binomial Negativa com ano e Virada (modelo D)')
ax2.plot(xx, np.exp(beta_A[0] + beta_A[1] * xx), color=CINZA, linewidth=2, linestyle='--',
         label='Poisson só com arrecadação (modelo A)')
ax2.plot(ks, taxas, 'o', color=VERMELHO, markersize=10, label='observado por grupo', zorder=3)

ax2.set_ylim(0, 2.2)
ax2.set_xlabel('Números ≤ 31 no sorteio')
ax2.set_ylabel('Ganhadores por R$ 100 milhões')
ax2.set_title('Cada número de calendário a mais no sorteio está associado a '
              f'{vir(rt, 2)} vezes a taxa de ganhadores\n'
              f'IC 95% [{vir(rt_lo, 2)}; {vir(rt_hi, 2)}] · os quatro ajustes ficam entre '
              f'{vir(min(rt_A, rt_B, rt_C, rt_D), 2)} e {vir(max(rt_A, rt_B, rt_C, rt_D), 2)}',
              fontsize=13, pad=15)
ax2.legend(frameon=False, fontsize=10.5, loc='upper left')
ax2.set_xticks(range(7))
ax2.text(1.6, 1.55,
         f'segundo o modelo D, de um número baixo para seis:\n{vir(salto, 1)} vezes a taxa esperada',
         fontsize=11, color=DOURADO, fontweight='bold', ha='center')

plt.figtext(0.5, 0.005, 'Modelo próprio · Binomial Negativa com deslocamento pela arrecadação, ano e Virada · '
            'banda: IC 95% da curva pela covariância completa, no ano médio | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-112-grafico-02-o-modelo.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · com quantos se divide (todos os 3.005 concursos) ══
fig3, ax3 = plt.subplots(figsize=(11, 6.5))
maxg = int(com.max())
cont = [int((com == i).sum()) for i in range(1, 8)]
cont.append(int((com >= 8).sum()))
rot = ['1', '2', '3', '4', '5', '6', '7', '8 ou mais']
cor3 = [VERDE] + [VERMELHO] * 7
barras = ax3.bar(rot, cont, color=cor3, alpha=0.85, width=0.6)
for b, v in zip(barras, cont):
    if v:
        ax3.text(b.get_x() + b.get_width() / 2, v + 4, mil(v), ha='center',
                 fontsize=11, fontweight='bold')
ax3.set_xlabel('Ganhadores no mesmo concurso')
ax3.set_ylabel('Número de concursos')
ax3.set_title('Na maioria das vezes em que alguém ganha, ganha sozinho\n'
              f'{mil(len(com))} concursos com ao menos um ganhador, de {mil(len(concursos))} · '
              f'{vir((g_todos == 1).sum() / len(com) * 100)}% tiveram um só',
              fontsize=13, pad=15)
ax3.text(4.2, max(cont) * 0.62,
         f'nos outros {vir((g_todos > 1).sum() / len(com) * 100)}% o prêmio foi partido\n'
         f'média de {vir(com.mean(), 2)} ganhadores quando há ganhador\n'
         f'o recorde foi {maxg}, na virada de 2018',
         fontsize=11, color=VERMELHO)

plt.figtext(0.5, 0.005, 'Fonte: Caixa Econômica Federal, concursos 1 a 3.005 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-112-grafico-03-quem-divide.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · a tradução em dinheiro ══
fig4, ax4 = plt.subplots(figsize=(11, 6.5))
fatias_N = [fatia_marginal(i, 'nb') * 100 for i in range(7)]
fatias_P = [fatia_marginal(i, 'poisson') * 100 for i in range(7)]
cor4 = [VERDE if i <= 1 else (DOURADO if i <= 4 else VERMELHO) for i in range(7)]
b4 = ax4.bar(range(7), fatias_N, color=cor4, alpha=0.88, width=0.6, label='Binomial Negativa')
ax4.plot(range(7), fatias_P, 'D', color='#333333', markersize=7, label='Poisson', zorder=3)
for i, (b, v) in enumerate(zip(b4, fatias_N)):
    ax4.text(b.get_x() + b.get_width() / 2, v + 1.2, vir(v) + '%', ha='center',
             fontsize=11, fontweight='bold')
ax4.set_xlabel('Números ≤ 31 na SUA aposta')
ax4.set_ylabel('Fatia esperada do prêmio, se você ganhar (%)')
ax4.set_ylim(0, 105)
ax4.set_title('O sorteio trata todas as combinações do mesmo jeito. Os apostadores não.\n'
              f'Fatia esperada, média sobre os {int(comum.sum())} concursos comuns da amostra, cada um com seu ano e volume',
              fontsize=13, pad=15)
ax4.legend(frameon=False, fontsize=10, loc='lower left')
ax4.annotate('', xy=(1.25, 62), xytext=(5.75, 62),
             arrowprops=dict(arrowstyle='->', color=VERMELHO, lw=2.2))
ax4.text(3.5, 65, f'cerca de {vir(ganho_N * 100, 0)}% a mais de prêmio esperado, se ganhar',
         ha='center', fontsize=12.5, color=VERMELHO, fontweight='bold')
ax4.text(3.5, 55, 'mesma chance de acertar as seis',
         ha='center', fontsize=10.5, color=CINZA)

plt.figtext(0.5, 0.005,
            'Modelo próprio · fatia esperada = E[1/(1+X)], X = outros ganhadores, sob Poisson e sob '
            'Binomial Negativa, média sobre os concursos | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-112-grafico-04-duas-apostas.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

O sorteio trata todas as combinações do mesmo jeito. Os apostadores não.

Quando a Mega-Sena sorteia números que cabem num calendário, aparece mais gente para
dividir o prêmio. Segundo o modelo, cada número ≤31 a mais no sorteio está associado a
**1,27 vez a taxa esperada de ganhadores** (IC 95% [1,17; 1,37]), controlando pela
arrecadação, pelo ano e pela Mega da Virada. Os quatro ajustes, de Poisson simples a
Binomial Negativa com tempo, ficam entre 1,27 e 1,38. De um número baixo para seis, o
modelo estima cerca de **3,3 vezes** a taxa esperada. São 1.932 concursos oficiais.

E aparece uma segunda marca compatível com a hipótese do calendário: números de 1 a 12
podem ser dia ou mês e, no modelo, estão associados a um efeito maior, 1,43 contra 1,17
dos números de 13 a 31. Não prova que sejam aniversários, mas é exatamente o padrão que
essa hipótese faria esperar.

Ninguém combinou nada. Milhões de pessoas simplesmente amam as mesmas datas.

E aí está o detalhe que muda a conta. Todas as combinações têm a mesma chance de sair,
e isso continua valendo. O que muda não é ganhar.

**É quanto sobra se ganhar.**

Na média dos concursos comuns da amostra, cada um com seu ano e seu volume, o modelo
estima uma fatia esperada de **quatro quintos** para uma combinação formada só por
números de calendário, contra **mais de nove décimos** para uma combinação com apenas
um. A mesma chance de acertar, cerca de **16% a mais de prêmio esperado, condicionado ao
acerto**, pela diferença de popularidade estimada entre as combinações. Sob Poisson e
sob Binomial Negativa, o mesmo resultado; num concurso pequeno o ganho fica em 9%, num
grande chega a 26%.

O aniversário do seu filho não é menos provável que qualquer outro número.
Ele só não é seu.

*Se a sorte for igual para todo mundo, o que ainda dá para escolher?*

---

### ⚠️ Limitações do Modelo

- **A base é cópia pública**, não o arquivo oficial baixado da Caixa. Mesma ressalva do
  **#104**. A origem está declarada porque a série dos cabalísticos não exige fonte
  revisada por pares, mas exige dizer de onde veio o dado.
- **Só entram no modelo os concursos com arrecadação declarada, de 2009 em diante**
  (1.932 de 3.005). A contagem de ganhadores existe para todos e é usada na descrição
  (652 concursos com ganhador, 490 com um só).
- **A exposição é a arrecadação em reais, não o número de combinações apostadas.** O
  modelo controla pelas diferenças entre os anos, não pelo preço da aposta diretamente.
  Como o preço é constante dentro de quase todo ano, o efeito de ano absorve a maior
  parte dessa mudança, mas não toda; uma exposição em "apostas simples equivalentes"
  exigiria a tabela histórica de preços com fonte, e apostas com mais de seis números
  continuariam sendo aproximação.
- **A Mega da Virada é identificada só pela data** (31/12, e a Virada de 2025, que este
  arquivo registra em 01/01/2026): 17 concursos. Arrecadação alta sozinha não marca
  Virada.
- **A reamostragem da camada 3 reajusta o modelo principal** (Binomial Negativa com ano
  e Virada) em cada uma das 500 reamostras, com partida quente no ajuste completo.
- **O grupo de 16 concursos sem nenhum número ≤31 destoa da tendência.** Ele fica no
  modelo principal. Como a estimativa é muito imprecisa, a camada 4 repete o modelo sem
  ele: o efeito permanece e fica maior. Um desses concursos foi 50-51-56-57-58-59, que
  é sequência, outro padrão que as pessoas jogam; é curiosidade, não explicação
  estatística do grupo.
- **O modelo mede ganhadores por combinação sorteada, não escolha de aposta.** Que a
  causa sejam datas é a explicação mais simples e bate com a fronteira do 31 e com o
  peso extra dos números até 12, mas o dado não vê o volante de ninguém. Por isso o
  texto fala em "popularidade estimada", não em "aniversário".
- **A superdispersão (φ = 1,53) diz que os ganhadores variam mais do que um Poisson
  simples explicaria.** É compatível com o que se esperaria se as apostas não se
  espalhassem igual pelas combinações (datas, sequências, desenhos no volante, bolões,
  concursos especiais), mas não identifica sozinha a causa. A Binomial Negativa a
  incorpora nas camadas 2, 3 e 6.
- **A fatia esperada depende do volume de apostas do concurso.** É uma previsão
  marginal: calculada concurso a concurso, com o ano e a arrecadação de cada um, e
  depois tirada a média sobre os concursos comuns. Concurso maior, mais chance de outro
  ganhador com a mesma combinação, fatia menor para todo mundo: o ganho relativo vai de
  9% no décimo inferior de arrecadação a 26% no décimo superior.
- **Evitar datas não aumenta a chance de ganhar.** Aumenta a fatia esperada se ganhar.
  A chance de acertar as seis continua uma em 50.063.860 para qualquer combinação,
  como o **#104** mostrou.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*